# Telecom Egypt Intelligent Assistant — ASR & RAG Pipeline Walkthrough

This notebook walks through the two core technical pieces of the assistant:

1. **RAG pipeline** — embedding, retrieval from Qdrant, and grounded answer generation
2. **ASR pipeline** — speech-to-text via `faster-whisper`, including a real bug found and fixed during development

Everything here calls the actual project modules (`rag/`, `ingestion/`, `voice/`) rather than reimplementing logic for the notebook — this is a walkthrough of the real system, not a simplified demo.

**Prerequisites to run this notebook:**
- Ollama running locally with `qwen2.5:7b` and `bge-m3` pulled
- Qdrant running (`docker run -d -p 6333:6333 qdrant/qdrant`) with the `TE.Eg` collection already populated (see README §3.6)
- Run from the project root so the `rag`, `ingestion`, and `voice` packages import correctly


## 1. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))  # adjust if running from a different working directory

from rag.generator import search, generate_answer
from rag.pipeline import answer_question
from ingestion.vector_store import QdrantStorage


## 2. RAG Pipeline — Retrieval

The retrieval step embeds the user's question with `bge-m3` and searches the Qdrant `TE.Eg` collection
for the most relevant chunks scraped from te.eg. Let's inspect retrieval in isolation first, before
involving the LLM — this is the same debugging technique used throughout development to separate
"did we retrieve the right content" from "did the model answer well given what it received."


In [ ]:
question = "What is the customer service number?"

result = search(question, top_k=5)

print(f"Retrieved {len(result['contexts'])} chunks\n")
for i, ctx in enumerate(result["contexts"]):
    print(f"--- Chunk {i} ---")
    print(ctx[:300])
    print()

print("Sources:", result["sources"])


**Expected output (verified during development):** the top chunks should include te.eg's
contact-us / live-chat pages, and the sources list should surface URLs like
`te.eg/en/about-te/contact-us`. This exact question was used repeatedly during debugging as a
known-good sanity check after any change to the ingestion pipeline.


## 3. RAG Pipeline — Generation

Generation takes the retrieved contexts and asks `qwen2.5:7b` (via Ollama, `temperature=0` for
factual grounding) to answer strictly from that context, matching the question's language, and
explicitly admitting when the context doesn't cover the question.


In [ ]:
answer = generate_answer(question, result["contexts"])
print(answer)


## 4. Full RAG Pipeline — End to End

`answer_question()` combines retrieval and generation, and short-circuits with a canned response
if retrieval finds nothing — this avoids sending an empty-context prompt to the LLM, which would
otherwise risk a hallucinated answer.


In [ ]:
result = answer_question("ما هي ارقام خدمه العملاء؟")

print("Answer:\n", result["answer"])
print("\nSources:", result["sources"])


### A bug found and fixed here, for the record

Earlier in development, this exact question returned the *business*-customer document list
instead of the individual-customer one. Root-causing it surfaced two separate, real bugs:

1. **Template-detection exclusivity bug in the scraper** — pages with both a main content
   section and an FAQ accordion were classified as "FAQ-only," silently dropping their primary
   content. Fixed by extracting all applicable content types per page independently. Verified via
   before/after record count: 398 → 432 records.
2. **Qdrant point-ID collisions** — FAQ/legal chunks don't get a meaningful `chunk_index` from the
   chunker (they're atomic units, not re-split), so an ID scheme based on `chunk_index` caused many
   distinct FAQ entries on the same page to silently overwrite each other in the vector store.
   Diagnosed via direct collision counting (263 of 639 chunks were being lost on upsert), fixed by
   hashing each chunk's actual text into its point ID instead.

Both fixes are in `scraper/extractor.py` and `ingestion/vector_store.py` respectively — see the
README for the full writeup.


## 5. Bilingual & Dialect Handling

The assistant is expected to handle Arabic (MSA), Egyptian dialect, and English. Below are the
same underlying question asked three ways, to demonstrate the retrieval + generation pipeline
is genuinely language-agnostic rather than English-first with translation bolted on.


In [ ]:
questions = {
    "English": "What is the customer service number?",
    "Arabic": "ما هو رقم خدمة العملاء؟",
    "Egyptian dialect": "ايه هو رقم خدمة العملاء؟",
}

for label, q in questions.items():
    r = answer_question(q)
    print(f"=== {label} ===")
    print(q)
    print("->", r["answer"])
    print()


**Verified during development:** all three phrasings correctly return the same grounded answer
(customer service numbers 111 / 01555000111 / 19777), each in the language the question was asked in.


## 6. ASR Pipeline — Speech to Text

`faster-whisper` (`large-v3`) transcribes audio with automatic language detection
(`language=None`), since the system doesn't know in advance whether a user will speak Arabic,
English, or Egyptian dialect.


In [ ]:
from voice.asr import transcribe

# Point this at any real audio file (English or Arabic) to test
audio_path = "path/to/your/test_audio.wav"

transcript, detected_language = transcribe(audio_path)
print("Transcript:", transcript)
print("Detected language:", detected_language)


### A bug found and fixed here, for the record

Short audio clips (~3 seconds) were sometimes misdetected. In one reproducible case, a clear
English question ("Hello, what is the customer service number?") was transcribed as Russian text
and, downstream, answered in Korean by the LLM — a compounding failure across two components
triggered by one root cause.

**Root cause:** Whisper's language detection is less reliable on short clips, especially with
silence/breath at the clip's edges skewing the detected language.

**Fix:** enabling voice-activity-detection filtering (`vad_filter=True`), which trims
silence/noise before transcription and language detection run.

**Before / after, same audio file:**

| | Transcript | Detected language |
|---|---|---|
| Before (`vad_filter=False`) | "Вот это сервис Customer Numbers" | `ru` |
| After (`vad_filter=True`) | "Hello, what is the customer service number?" | `en` |


In [ ]:
# Demonstrating the fix directly: vad_filter=True vs False on the same file
from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="int8_float16")  # switch to "cpu"/"int8" if no GPU

for use_vad in [False, True]:
    segments, info = model.transcribe(audio_path, language=None, beam_size=5, vad_filter=use_vad)
    text = " ".join(s.text.strip() for s in segments)
    print(f"vad_filter={use_vad}: '{text}' (detected: {info.language})")


## 7. Full Voice Round Trip

Putting it together: audio in, transcribed text, retrieved + generated answer, synthesized audio out.
This mirrors exactly what the `/voice` FastAPI endpoint does — see `app/routes/voice.py`.


In [ ]:
from voice.tts import synthesize

transcript, detected_language = transcribe(audio_path)
print("1. Transcript:", transcript)

result = answer_question(transcript)
print("2. Answer:", result["answer"])

tts_result = synthesize(result["answer"])
print("3. Synthesized audio saved to:", tts_result.path)


## 8. Document Upload — Session-Scoped Retrieval

Uploaded documents are embedded and tagged with a `session_id`, then filtered at query time so
one user's uploaded document is never retrievable by another user's questions — only the shared
te.eg knowledge base is common across sessions. This cell demonstrates the filtering directly.


In [ ]:
from ingestion.user_loader import chunk_uploaded_document
from ingestion.embedder import embed_chunks

store = QdrantStorage()
session_id = "demo-session-123"

# chunks = chunk_uploaded_document("path/to/a/sample.pdf", session_id)
# embedded = embed_chunks(chunks)
# store.upsert_chunks(embedded)

# A query with this session_id will see both the shared KB and this session's upload;
# a query with no session_id (or a different one) will only see the shared KB.
result_with_session = search("some question about the uploaded doc", top_k=5, session_id=session_id)
result_without_session = search("some question about the uploaded doc", top_k=5, session_id=None)

print("With session:", len(result_with_session["contexts"]), "contexts")
print("Without session:", len(result_without_session["contexts"]), "contexts")


## 9. Summary

| Component | Status |
|---|---|
| Text chat + citations |  Verified |
| Voice (ASR → RAG → TTS) | Verified, in English + Arabic MSA + Egyptian dialect |
| Document upload, session-scoped | Verified |
| Bilingual / dialect handling | Verified |
| Data quality (scraper + vector store) |  Two real bugs found, root-caused, and fixed with before/after evidence |

See the accompanying README for full setup instructions, architecture, and known limitations.
